# 00 — Data Ingestion & Validation

Loads the full unfiltered Scopus export, validates schema and coding fields, applies the language filter (English-only), and exports the clean analysis corpus to `data/processed/corpus_clean.parquet`.

**All downstream notebooks load from `corpus_clean.parquet` — do not skip this notebook.**

In [68]:
# ── CONFIG ────────────────────────────────────────────────────────────────────
RAW_EXCEL     = "../data/raw/export.xlsx"
PROCESSED_DIR = "../data/processed/"
OUTPUTS_DIR   = "../outputs/"
SHEET_DATA    = "Screened"
SHEET_CB      = "Codebook"
SHEET_PRISMA  = "PRISMA Log"

In [69]:
# ── LOAD ──────────────────────────────────────────────────────────────────────
import sys
sys.path.insert(0, "..")

from pathlib import Path
import warnings
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from IPython.display import display

from scripts.utils import load_raw, assign_phase, PATHS

# Create output dirs
Path(PROCESSED_DIR).mkdir(parents=True, exist_ok=True)
Path(OUTPUTS_DIR).mkdir(parents=True, exist_ok=True)

df_raw    = load_raw()   # full 455 records, Include + Exclude
df_prisma = pd.read_excel(RAW_EXCEL, sheet_name=SHEET_PRISMA)
df_cb     = pd.read_excel(RAW_EXCEL, sheet_name=SHEET_CB)

print(f"Loaded {len(df_raw):,} total records × {len(df_raw.columns)} columns")
print(f"Columns: {list(df_raw.columns)}")

Loaded 979 total records × 23 columns
Columns: ['Record_ID', 'Screening_Decision', 'Exclusion_Reason', 'Relevance Level', 'Contribution Type', 'Study Approach', 'Coding_Notes', 'Title', 'Year', 'Authors', 'Source title', 'Document Type', 'DOI', 'Link', 'Abstract', 'Author Keywords', 'Index Keywords', 'EID', 'Cited_by', 'Affiliations', 'Countries', 'Language', 'Source']


## 1 · Schema Validation

In [70]:
EXPECTED_COLS = [
    "Record_ID", "Screening_Decision", "Relevance Level", "Contribution Type",
    "Study Approach", "Coding_Notes", "Title", "Year", "Authors",
    "Source title", "Document Type", "DOI", "Link", "Abstract",
    "Author Keywords", "Index Keywords", "EID", "Cited_by",
    "Affiliations", "Countries", "Language",
]

missing_cols = [c for c in EXPECTED_COLS if c not in df_raw.columns]
extra_cols   = [c for c in df_raw.columns if c not in EXPECTED_COLS]

assert not missing_cols, f"Missing columns: {missing_cols}"
print(f"✓ All {len(EXPECTED_COLS)} expected columns present")
if extra_cols:
    print(f"  Extra columns (ignored): {extra_cols}")

# dtype checks
schema = pd.DataFrame({
    "column": df_raw.columns,
    "dtype":  df_raw.dtypes.values,
    "non_null": df_raw.notna().sum().values,
    "null":     df_raw.isna().sum().values,
    "null_pct": (df_raw.isna().mean() * 100).round(1).values,
})
display(schema)

✓ All 21 expected columns present
  Extra columns (ignored): ['Exclusion_Reason', 'Source']


,column,dtype,non_null,null,null_pct
0,Record_ID,int64,979,0,0.0
1,Screening_Decision,str,979,0,0.0
2,Exclusion_Reason,str,596,383,39.1
3,Relevance Level,str,576,403,41.2
4,Contribution Type,str,527,452,46.2
5,Study Approach,str,527,452,46.2
6,Coding_Notes,str,576,403,41.2
7,Title,str,979,0,0.0
8,Year,int64,979,0,0.0
9,Authors,str,954,25,2.6


## 2 · Screening Decision Overview

In [71]:
decision_counts = df_raw["Screening_Decision"].value_counts(dropna=False)
total = len(df_raw)
print("Screening_Decision distribution (full export):")
for val, n in decision_counts.items():
    print(f"  {val:10s}  N={n:4d}  ({100*n/total:.1f}%)")
print(f"  {'TOTAL':10s}  N={total:4d}")

Screening_Decision distribution (full export):
  Exclude     N= 597  (61.0%)
  Include     N= 382  (39.0%)
  TOTAL       N= 979


## 3 · Language Validation & English-Only Filter

All records with `Screening_Decision == 'Include'` must be English for the analysis corpus. Non-English included records are excluded here (language cleaning phase).

In [72]:
print("Language distribution by Screening_Decision:")
lang_tab = df_raw.groupby("Screening_Decision")["Language"].value_counts(dropna=False)
display(lang_tab.rename("N").reset_index())

# Included records split by language
df_included = df_raw[df_raw["Screening_Decision"] == "Include"]
included_en  = df_included[df_included["Language"] == "English"]
included_non = df_included[df_included["Language"] != "English"]

print(f"\nIncluded records total: {len(df_included)}")
print(f"  → English:     {len(included_en)}  (kept for analysis)")
print(f"  → Non-English: {len(included_non)}  (removed — language cleaning)")

if len(included_non) > 0:
    print("\nRemoved non-English included records:")
    display(included_non[["Record_ID", "Language", "Title", "Year"]])

Language distribution by Screening_Decision:


,Screening_Decision,Language,N
0,Exclude,English,591
1,Exclude,Portuguese,4
2,Exclude,Chinese,1
3,Exclude,Turkish,1
4,Include,English,375
5,Include,Chinese,6
6,Include,French,1



Included records total: 382
  → English:     375  (kept for analysis)
  → Non-English: 7  (removed — language cleaning)

Removed non-English included records:


,Record_ID,Language,Title,Year
69,70,Chinese,Research on the equal card force competition s...,2021
160,161,Chinese,A discussion on a new-typed digital media secu...,2007
168,169,Chinese,Research on the collusion problem for MMOG,2010
186,187,Chinese,Cheating detection mechanism based on trust ma...,2008
209,210,Chinese,A cheat-detection method based on fuzzy synthe...,2010
663,664,Chinese,An Anti-cheat Method of Game Based on Windows ...,2020
804,805,French,Playing with Frameworks of Experience: Scams a...,2020


## 4 · Allowed-Value Validation

In [73]:
# Validate on the full dataset (Include + Exclude)
ALLOWED = {
    "Screening_Decision": {"Include", "Exclude"},
    "Relevance Level":    {"core", "relevant", "peripheral", "borderline"},
    "Contribution Type":  {"technical", "socio_behavioral", "conceptual_taxonomic",
                           "legal_governance", "review_survey", "peripheral", None},
    "Study Approach":     {"detection", "prevention", "conceptualization",
                           "behavioral_explanation", "infrastructure_support",
                           "governance", "measurement", "offensive",
                           "not_applicable", None},
}

for field, allowed in ALLOWED.items():
    found = set(df_raw[field].unique())
    unexpected = found - {v for v in allowed if v is not None} - {np.nan, None}
    missing_in_data = {v for v in allowed if v is not None} - found
    status = "✓" if not unexpected else "⚠"
    print(f"{status} {field}")
    print(f"    Found:           {sorted(str(v) for v in found if pd.notna(v))}")
    if unexpected:
        print(f"    ⚠ Unexpected:   {unexpected}")
    if missing_in_data:
        print(f"    ℹ Not in data:  {missing_in_data}  (defined in codebook but absent)")

✓ Screening_Decision
    Found:           ['Exclude', 'Include']
✓ Relevance Level
    Found:           ['borderline', 'core', 'peripheral', 'relevant']
⚠ Contribution Type
    Found:           ['conceptual_taxonomic', 'legal_governance', 'other_unclear', 'peripheral', 'review_survey', 'socio_behavioral', 'technical']
    ⚠ Unexpected:   {'other_unclear'}
✓ Study Approach
    Found:           ['behavioral_explanation', 'conceptualization', 'detection', 'governance', 'infrastructure_support', 'measurement', 'not_applicable', 'offensive', 'prevention']


## 5 · Missing Value Audit

In [74]:
import plotly.express as px

null_summary = (
    included_en.isnull()
    .sum()
    .rename("N_null")
    .reset_index()
    .rename(columns={"index": "column"})
)
null_summary["pct"] = (null_summary["N_null"] / len(included_en) * 100).round(1)
null_summary = null_summary[null_summary["N_null"] > 0].sort_values("pct", ascending=False)

print(f"Fields with missing values ({len(null_summary)} of {len(included_en.columns)}):")
display(null_summary)

fig = px.bar(
    null_summary,
    x="pct",
    y="column",
    orientation="h",
    text=null_summary.apply(lambda r: f"N={r.N_null} ({r.pct}%)", axis=1),
    title="Missing Value Rate by Field (full export, N=455)",
    labels={"pct": "Missing (%)", "column": ""},
    color_discrete_sequence=["#546E7A"],
)
fig.update_traces(textposition="outside")
fig.update_layout(xaxis_range=[0, null_summary["pct"].max() * 1.25], height=400)
fig.show()

Fields with missing values (10 of 23):


,column,N_null,pct
2,Exclusion_Reason,372,99.2
16,Index Keywords,85,22.7
15,Author Keywords,82,21.9
12,DOI,46,12.3
13,Link,21,5.6
17,EID,17,4.5
20,Countries,9,2.4
19,Affiliations,8,2.1
14,Abstract,2,0.5
18,Cited_by,1,0.3


## 6 · PRISMA Funnel Verification

In [ ]:
print("PRISMA Log:")
display(df_prisma)

# Key counts — updated to match new export
n_identified   = df_prisma.iloc[0, 1]  # Initially identified records
n_dedup        = df_prisma.iloc[1, 1]   # duplicates removed
n_lang_removed = df_prisma.iloc[2, 1]   # non-English removed before screening (5 Include Chinese + 3 Exclude non-English)
n_screened     = df_prisma.iloc[3, 1]   # screened records
n_included     = df_prisma.iloc[4, 1]   # English included records
n_excluded     = df_prisma.iloc[5, 1]   # excluded after screening

print(f"\nPRISMA arithmetic: {n_identified}−{n_dedup} (dedup)−{n_lang_removed} (language) = {n_screened} screened")
print(f"Screened: {n_included} included + {n_excluded} excluded = {n_included + n_excluded}")
assert n_included + n_excluded == n_screened, "PRISMA counts don't add up!"
print("✓ PRISMA counts consistent")

print(f"\nActual file: {len(df_raw)} total rows  ({decision_counts.get('Include',0)} Include / {decision_counts.get('Exclude',0)} Exclude)")
print(f"After English filter: {len(included_en)} analysis corpus")

PRISMA Log:


,PRISMA item,Current count,Notes
0,Records identified from uploaded Scopus CSV,979,Screening complete from differents sources (Sc...
1,Duplicate records removed,419,Remove duplicated or extended to journal
2,Records removed before screening,13,Removed from corpus all not-english texts
3,Records screened so far,548,All records screened;
4,Records included so far,375,Final provisional includes after screening all...
5,Records excluded so far,173,Final provisional exclusions after screening a...
6,Records remaining to screen,0,Screening complete.
7,Final SLM corpus,375,Provisional final SLM corpus before duplicate ...



PRISMA arithmetic: 979−419 (dedup)−13 (language) = 548 screened
Screened: 375 included + 173 excluded = 548


AssertionError: PRISMA counts don't add up!

In [76]:
# PRISMA Sankey — Identified flows to: Duplicates removed, Language removed, Screened
# Screened flows to: Included, Excluded
from scripts.utils import save_figure

node_labels = [
    f"Identified<br>(N={n_identified})",
    f"Duplicates<br>removed (N={n_dedup})",
    f"Language<br>filtered (N={n_lang_removed})",
    f"Screened<br>(N={n_screened})",
    f"Included<br>(English, N={n_included})",
    f"Excluded<br>(N={n_excluded})",
]
node_colors = ["#1565C0", "#C62828", "#E65100", "#1565C0", "#2E7D32", "#C62828"]

# 0=Identified, 1=Duplicates, 2=Language removed, 3=Screened, 4=Included, 5=Excluded
src = [0,       0,                0,              3,          3        ]
tgt = [1,       2,                3,              4,          5        ]
val = [n_dedup, n_lang_removed,   n_screened,     n_included, n_excluded]
lbl = [str(v) for v in val]
clr = ["#FFCDD2", "#FFE0B2", "#BBDEFB", "#C8E6C9", "#FFCDD2"]

fig_prisma = go.Figure(go.Sankey(
    arrangement="snap",
    node=dict(label=node_labels, color=node_colors, pad=20, thickness=20),
    link=dict(source=src, target=tgt, value=val, label=lbl, color=clr,
              hovertemplate="%{source.label} → %{target.label}<br>N=%{value}<extra></extra>"),
))
fig_prisma.update_layout(
    #title_text="PRISMA Funnel — SLM Anti-Cheat Study",
    font_size=13,
    height=500,
    margin=dict(r=200),
)
fig_prisma.show()
save_figure(fig_prisma, "00_prisma_sankey")

## 7 · Clean & Filter Corpus

In [77]:
# Filter: Include + English
df_clean = (
    df_raw[
        (df_raw["Screening_Decision"] == "Include") &
        (df_raw["Language"] == "English")
    ]
    .copy()
    .reset_index(drop=True)
)

# Normalize column names
df_clean = df_clean.rename(columns={
    "Relevance Level":  "relevance_level",
    "Contribution Type": "contribution_type",
    "Study Approach":   "study_approach",
    "Source title":     "source_title",
    "Document Type":    "document_type",
    "Author Keywords":  "author_keywords",
    "Index Keywords":   "index_keywords",
    "Coding_Notes":     "coding_notes",
})

# Cast types
df_clean["Year"]     = df_clean["Year"].astype(int)
df_clean["Cited_by"] = pd.to_numeric(df_clean["Cited_by"], errors="coerce").fillna(0).astype(int)

# Strip whitespace from all string columns
str_cols = df_clean.select_dtypes(include="str").columns
for c in str_cols:
    df_clean[c] = df_clean[c].str.strip()

# Add phase column
df_clean["phase"] = assign_phase(df_clean["Year"])

assert len(df_clean) == 329, f"Expected 329 records, got {len(df_clean)}"
print(f"✓ Clean corpus: {len(df_clean)} records")
print(f"  Columns: {list(df_clean.columns)}")
print(f"  Phase distribution:")
print(df_clean["phase"].value_counts().to_string())

AssertionError: Expected 329 records, got 375

## 8 · Export

In [ ]:
out_path = PROCESSED_DIR + "corpus_clean.parquet"
df_clean.to_parquet(out_path, index=False)

import os
size_kb = os.path.getsize(out_path) / 1024
print(f"✓ Exported corpus_clean.parquet  ({len(df_clean)} rows × {len(df_clean.columns)} cols, {size_kb:.1f} KB)")
print(f"  Path: {out_path}")

✓ Exported corpus_clean.parquet  (329 rows × 23 cols, 350.7 KB)
  Path: ../data/processed/corpus_clean.parquet
